# Aula 09 — Laboratório: busca aleatória e AutoML

**Consultoria Especializada em Ciência de Dados 1**

Na Aula 08 a grade testou todas as combinações que você escreveu. Hoje o orçamento passa a ser
**seu**: você diz quantas medições quer (`n_iter`) e sorteia onde medir. O laboratório treina as
três decisões que isso cria — **qual distribuição**, **qual faixa** e **quantos sorteios** — e
termina no AutoML, que faz a mesma coisa para o modelo inteiro.

A base é a mesma da Aula 08 (`5_musicas.csv`), com o mesmo alvo (`hit`), o mesmo Pipeline e o
mesmo teste guardado. O que muda é a busca.

Escreva as respostas nas células indicadas. Toda afirmação que menciona um número deve trazer o
número que saiu no seu notebook.

---
## Parte 0 — Ambiente, dados e o que já está montado

Nada aqui é novo: é o fecho da Aula 08. A coluna `popularidade` fica de fora (ela vaza o alvo —
voltamos a isso no Exercício 10) e o teste de 30 % fica guardado até o Exercício 8.

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import uniform, loguniform, randint
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     GridSearchCV, RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

mus = pd.read_csv("5_musicas.csv")
y = mus["hit"]
X = pd.get_dummies(mus.drop(columns=["id", "titulo", "artista", "popularidade", "hit"]),
                   columns=["genero"])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

def pipe():
    return Pipeline([("esc", StandardScaler()),
                     ("clf", RandomForestClassifier(n_estimators=100, random_state=0))])

print("X:", X.shape, "| hits:", int(y.sum()), f"({y.mean():.1%})")
print("treino:", len(X_tr), "| teste:", len(X_te), "| hits no teste:", int(y_te.sum()))
print("métrica da aula: average_precision (AP)")

---
## Parte 1 — Por que sortear, antes de sortear

Duas contas decidem a aula inteira: quanto custa cada caminho, e qual a chance de o sorteio
achar a região boa.

#### Exemplo

**Exemplo.** O custo da grade é o produto do número de valores de cada hiperparâmetro, vezes
as dobras. Com 2 valores de um e 3 de outro, e 5 dobras, são 2 × 3 × 5 = 30 ajustes.

In [ ]:
valores = [2, 3]
combos = np.prod(valores)
print("combinações:", combos, "| com cv = 5:", combos * 5, "ajustes")

### Exercício 1 — o custo de cada caminho
A floresta tem 5 hiperparâmetros. Uma grade modesta usa 4, 5, 4, 4 e 4 valores. Calcule:

1. quantas combinações a grade gera e quantos ajustes isso custa com `cv = 5`;
2. quantos ajustes custa uma `RandomizedSearchCV` com `n_iter = 40` e `cv = 5`;
3. quantos ajustes a grade custaria se você acrescentasse **um sexto** hiperparâmetro com 4
   valores — e quantos custaria a busca aleatória no mesmo caso.

In [ ]:
# TODO — combinações da grade e ajustes com cv = 5
# TODO — ajustes da busca aleatória com n_iter = 40
# TODO — repita as duas contas com um sexto hiperparâmetro de 4 valores

**Resposta:**

#### Exemplo

**Exemplo.** Se a região boa ocupa uma fração `p` do espaço, cada sorteio erra com
probabilidade `1 − p`, e `n` sorteios erram todos com `(1 − p)ⁿ`. A chance de acertar ao menos
uma vez é o complemento. Com p = 10 % e n = 10 sorteios:

In [ ]:
p, n = 0.10, 10
print(f"chance de errar todos: (1 - {p})^{n} = {(1-p)**n:.4f}")
print(f"chance de acertar ao menos um: {1 - (1-p)**n:.4f}")

### Exercício 2 — a chance de o sorteio achar a região boa
Monte a tabela da chance `1 − (1 − p)ⁿ` para `p` em `[0.05, 0.10, 0.15, 0.25]` e `n` em
`[10, 16, 30, 60]`, em porcentagem.

Depois responda: com `p = 15 %`, quantos sorteios você precisa para passar de 90 %? E o que
acontece com essa conta se o espaço tiver 2, 5 ou 50 hiperparâmetros?

In [ ]:
# TODO — tabela de 1 - (1 - p)**n para cada p e cada n
# TODO — ache o menor n que passa de 90% com p = 0,15

**Resposta:**

---
## Parte 2 — RandomizedSearchCV na prática

Três decisões: qual **distribuição**, qual **faixa**, quantos **sorteios**.

#### Exemplo

**Exemplo.** O dicionário da busca aleatória não leva listas de valores, leva **distribuições**
— objetos com `.rvs()`. Sortear 5 valores de `uniform(0, 1)`:

In [ ]:
print(np.round(uniform(0, 1).rvs(5, random_state=0), 3))

### Exercício 3 — as distribuições do dicionário
Sorteie 2.000 valores de cada uma e imprima o **mínimo e o máximo** de cada amostra:

- `uniform(0.1, 0.8)`
- `randint(100, 601)`
- `loguniform(0.01, 100)`

Para a loguniform, imprima também **que fração dos sorteios caiu em cada ordem de grandeza**
(0,01–0,1 · 0,1–1 · 1–10 · 10–100) e compare com o que daria `uniform(0.01, 100)`.

Responda: qual é a faixa real de `uniform(0.1, 0.8)`? E por que o `C` da logística (0,01 a 100)
se sorteia com loguniform?

In [ ]:
# TODO — sorteie 2000 valores de cada distribuição e imprima mínimo e máximo
# TODO — para a loguniform, conte a fração em cada ordem de grandeza
# TODO — faça a mesma contagem com uniform(0.01, 100) para comparar

**Resposta:**

#### Exemplo

**Exemplo.** Os dois recebem o mesmo pipeline, o mesmo `cv` e a mesma métrica. Muda o que
entra no segundo argumento: a grade recebe **listas**, a busca aleatória recebe **distribuições**
e um `n_iter`. Primeiro a grade, com 4 × 4 = 16 combinações:

In [ ]:
t = time.time()
grade = {"clf__ccp_alpha":   [1e-4, 1e-3, 1e-2, 1e-1],
         "clf__max_features": [0.2, 0.4, 0.6, 0.8]}
gs = GridSearchCV(pipe(), grade, cv=cv, scoring="average_precision", n_jobs=-1).fit(X_tr, y_tr)
print(f"grade: {len(gs.cv_results_['mean_test_score'])} combinações | "
      f"melhor AP {gs.best_score_:.4f} | {time.time()-t:.0f} s")
print("melhor:", gs.best_params_)

### Exercício 4 — a grade e o sorteio, com o mesmo orçamento
Rode uma `RandomizedSearchCV` com o **mesmo orçamento** (16 medições) sobre os mesmos dois
hiperparâmetros, usando `loguniform(1e-5, 1e-1)` para o `ccp_alpha` e `uniform(0.1, 0.8)` para o
`max_features`, com `random_state=0`.

Compare a melhor AP com a da grade e responda: quantos valores **distintos** de `ccp_alpha` cada
caminho testou? É daí que vem a diferença?

In [ ]:
# TODO — monte o dicionário de distribuições
# dist = {...}
# TODO — RandomizedSearchCV com n_iter=16, cv=cv, scoring="average_precision", random_state=0
# TODO — compare best_score_ com o da grade e conte os valores distintos de cada caminho

**Resposta:**

#### Exemplo

**Exemplo.** Uma busca aleatória é uma amostra: trocar a semente troca os pontos sorteados e,
com eles, o resultado. Antes de comemorar o 0,627, vale ver quanto ele balança. Com três
sementes:

In [ ]:
for s in [0, 1, 2]:
    b = RandomizedSearchCV(pipe(), dist, n_iter=16, cv=cv, scoring="average_precision",
                           random_state=s, n_jobs=-1).fit(X_tr, y_tr)
    print(f"semente {s}: {b.best_score_:.4f}")

### Exercício 5 — foi sorte da semente?
Repita a busca com **20 sementes** (`range(20)`), guarde os 20 melhores escores e imprima a
média, o mínimo e o máximo, além da **fração das sementes que ficou acima da grade**
(`gs.best_score_`).

Responda: o 0,627 da semente 0 foi sorte? E o que você reportaria num relatório — o melhor
resultado ou a distribuição?

In [ ]:
# TODO — rode a busca para random_state em range(20) e guarde os best_score_
# TODO — imprima média, mínimo, máximo e a fração acima de gs.best_score_

**Resposta:**

#### Exemplo

**Exemplo.** O `best_params_` é **uma linha** do `cv_results_`. Olhar só para ela esconde se o
primeiro lugar ganhou por uma diferença real ou por ruído. O `cv_results_` é um dicionário que
vira DataFrame — veja as três primeiras colocações da busca do Exercício 4:

In [ ]:
res = pd.DataFrame(rs.cv_results_)
cols = ["param_clf__ccp_alpha", "param_clf__max_features", "mean_test_score", "std_test_score"]
print(res.sort_values("rank_test_score")[cols].head(3).round(4).to_string(index=False))

### Exercício 6 — o pódio e o platô
Monte a tabela das **5 primeiras colocações** com a média e o desvio entre dobras, e calcule:

1. a diferença entre o 1º e o 5º lugar;
2. o desvio entre dobras do 1º lugar.

Responda: isso é um pódio ou um platô? E, se for platô, qual configuração você entregaria?

In [ ]:
# TODO — top 5 do cv_results_, com mean_test_score e std_test_score
# TODO — a diferença 1º - 5º e o desvio do 1º

**Resposta:**

#### Exemplo

**Exemplo.** Quando o melhor valor encontrado é exatamente o **extremo** da faixa que você
definiu, a busca foi até onde podia e parou na parede — o pico pode estar do lado de fora.
Veja onde caiu o melhor `ccp_alpha` em relação à faixa sorteada:

In [ ]:
melhor = rs.best_params_["clf__ccp_alpha"]
print(f"faixa definida : 1e-05 a 1e-01")
print(f"melhor valor   : {melhor:.2e}")
print(f"posição na faixa (em log): "
      f"{(np.log10(melhor) - np.log10(1e-5)) / (np.log10(1e-1) - np.log10(1e-5)):.0%}")

### Exercício 7 — o ótimo na borda da faixa
Rode uma busca com a faixa **estreitada de propósito** — `loguniform(1e-2, 1e-1)` para o
`ccp_alpha`, mantendo o resto igual — e veja onde cai o melhor valor e qual AP ela alcança.

Depois amplie a faixa para `loguniform(1e-7, 1e-1)` e rode de novo. Responda: o que o valor
colado na borda indicava, e o que mudou ao ampliar?

In [ ]:
# TODO — busca com ccp_alpha em loguniform(1e-2, 1e-1); onde caiu o melhor valor? qual AP?
# TODO — busca com ccp_alpha em loguniform(1e-7, 1e-1); compare

**Resposta:**

---
## Parte 3 — A nota honesta

Da mesma busca saem três números. Só dois podem ser reportados.

#### Exemplo

**Exemplo.** O `best_score_` é o **máximo** de n_iter medições feitas na mesma validação que
escolheu — por construção, otimista. A nested CV roda a busca **inteira** dentro de cada dobra de
fora, então nenhuma escolha vê a dobra que a mede. Veja a diferença dos dois em miniatura, com
3 dobras de fora e n_iter = 5:

In [ ]:
t = time.time()
mini = RandomizedSearchCV(pipe(), dist, n_iter=5, cv=cv, scoring="average_precision",
                          random_state=0, n_jobs=-1)
nested_mini = cross_val_score(mini, X_tr, y_tr, scoring="average_precision",
                              cv=StratifiedKFold(3, shuffle=True, random_state=1))
print(f"best_score_ (ingênuo): {mini.fit(X_tr, y_tr).best_score_:.4f}")
print(f"nested CV (3 dobras) : {nested_mini.mean():.4f} ± {nested_mini.std():.4f}")
print(f"({time.time()-t:.0f} s)")

### Exercício 8 — os três números da mesma busca
Produza os **três** números da busca do Exercício 4 (n_iter = 16):

1. o `best_score_`;
2. a **nested CV** com 5 dobras de fora (`StratifiedKFold(5, shuffle=True, random_state=1)`) —
   média e desvio;
3. a AP no **teste guardado** (`X_te`, `y_te`), que até agora não foi tocado.

Responda: qual deles **não** pode ser reportado como desempenho, e por quê? E quantos hits o
teste tem — isso muda a sua confiança no terceiro número?

In [ ]:
# TODO — best_score_ da busca do Exercício 4
# TODO — nested CV: cross_val_score(busca, X_tr, y_tr, cv=StratifiedKFold(5, ...))
# TODO — AP no teste guardado, com average_precision_score e predict_proba
# TODO — quantos hits o teste tem?

**Resposta:**

---
## Parte 4 — Um mini-AutoML

A mesma busca, agora escolhendo também **qual modelo**.

#### Exemplo

**Exemplo.** Para escolher também o **modelo**, o `RandomizedSearchCV` aceita uma **lista** de
dicionários. Cada dicionário fixa um `clf` e descreve só os hiperparâmetros daquele modelo — o
espaço fica **condicional**: o `ccp_alpha` só existe se o sorteio cair na floresta. Com dois
modelos e 6 sorteios:

In [ ]:
from sklearn.linear_model import LogisticRegression

espaco_mini = [
    {"clf": [LogisticRegression(max_iter=2000)], "clf__C": loguniform(0.01, 100)},
    {"clf": [RandomForestClassifier(n_estimators=100, random_state=0)],
     "clf__max_features": uniform(0.1, 0.8)},
]
b = RandomizedSearchCV(pipe(), espaco_mini, n_iter=6, cv=cv, scoring="average_precision",
                       random_state=0, n_jobs=-1).fit(X_tr, y_tr)
nomes = [type(p["clf"]).__name__ for p in b.cv_results_["params"]]
print("modelos sorteados:", nomes)
print("vencedor:", type(b.best_params_["clf"]).__name__, "| AP", round(b.best_score_, 4))

### Exercício 9 — o espaço condicional do AutoML
Monte um espaço com **três** modelos — logística (`C`), árvore de decisão (`max_depth`,
`min_samples_leaf`) e floresta (`max_features`, `ccp_alpha`) — e rode com `n_iter = 30`.

Depois monte uma tabela com, **para cada modelo**: quantos sorteios ele recebeu, a melhor AP e a
**mediana** das APs dele. Responda: o que pesou mais — escolher o modelo ou afinar o vencedor?

In [ ]:
# TODO — importe DecisionTreeClassifier e monte a lista de 3 dicionários
# TODO — RandomizedSearchCV com n_iter=30, random_state=0
# TODO — para cada modelo: nº de sorteios, melhor AP e mediana das APs

**Resposta:**

#### Exemplo

**Exemplo.** Com uma lista de modelos, o `n_iter` é dividido entre eles — e cada modelo recebe
só uma fatia. Com 30 sorteios e 3 modelos, a média é 10 por modelo; com 10 modelos, seriam 3.
Volte à conta do Exercício 2 e veja o que 3 sorteios garantem:

In [ ]:
for n in [3, 10, 30]:
    print(f"{n:>2} sorteios, p = 15%: {100*(1-0.85**n):.0f}% de chance de achar a região boa")

### Exercício 10 — o orçamento dividido
Rode o mesmo espaço de três modelos com `n_iter` em `[10, 30, 60]` e, para cada orçamento,
imprima a melhor AP, o modelo vencedor, quantos sorteios cada modelo recebeu e o tempo.

Responda: o que acontece com a melhor AP quando o orçamento triplica? E o que aconteceria se
você acrescentasse sete modelos ao espaço mantendo `n_iter = 30`?

In [ ]:
# TODO — para n_iter em [10, 30, 60], rode e guarde: melhor AP, vencedor, sorteios por modelo, tempo

**Resposta:**

#### Exemplo

**Exemplo.** O AutoML otimiza o que você deixar no `X`. Se uma coluna carrega informação que
não existiria na hora de prever, ele a usará com prazer — e o número fica lindo. Na base das
músicas, a coluna `popularidade` é medida **depois** de a faixa virar hit. Veja a correlação
dela com o alvo:

In [ ]:
print("correlação de popularidade com hit:", round(mus["popularidade"].corr(mus["hit"]), 3))
print("popularidade média | hits:", round(mus.loc[mus.hit == 1, "popularidade"].mean(), 1),
      "| não-hits:", round(mus.loc[mus.hit == 0, "popularidade"].mean(), 1))

### Exercício 11 — o que o AutoML não automatiza
Monte um `X_vaza` **incluindo** a coluna `popularidade`, refaça o corte treino/teste com a mesma
semente e rode a mesma busca de três modelos com `n_iter = 30`. Compare a melhor AP com a do
Exercício 9.

Responda: o AutoML avisou que havia algo errado? Quem teria de perceber, e como?

In [ ]:
# TODO — X_vaza = get_dummies de mus sem id/titulo/artista/hit (mantendo popularidade)
# TODO — novo train_test_split com random_state=0, stratify=y
# TODO — mesma busca de 3 modelos, n_iter=30; compare a AP

**Resposta:**

---
## Parte 5 — Síntese

#### Exemplo

**Exemplo.** Uma conclusão fechada tem três partes: a afirmação, o número que a sustenta e a
consequência prática. Por exemplo: *"com o mesmo orçamento de 16 medições, o sorteio deu 0,627
contra 0,615 da grade, e 20 em 20 sementes bateram a grade; por isso passo a usar a aleatória
como padrão quando há mais de dois hiperparâmetros."*

### Exercício 12 — três conclusões
Escreva **três conclusões** no formato do exemplo — afirmação, número e consequência. Sugestões
de tema: por que o custo da aleatória não depende do número de hiperparâmetros; o que você passou
a olhar no `cv_results_` além da primeira linha; qual número você reportaria no relatório e qual
ficaria no caderno; o que o AutoML resolveu e o que ele deixou na sua mão.

**Resposta:**

---
**Fim.** Você comparou grade e sorteio com o mesmo orçamento, montou o dicionário de
distribuições, leu o pódio e o platô, reagiu a um ótimo na borda, separou os três números da nota
honesta e rodou um mini-AutoML com espaço condicional — inclusive o caso em que ele otimiza
alegremente uma coluna que vaza o alvo.

No **Desafio 3** o que vale não é produzir o número, é justificá-lo.